## Overview

Summarize Text with Langchain

To summarize text using **LangChain**, you can use the `load_summarize_chain` function, which supports three main chain types: **`stuff`**, **`map_reduce`**, and **`refine`**. Each method handles long documents differently based on the LLM's context window.

- **`stuff`**: Best for shorter texts that fit within the model’s context window. It combines all input text into a single prompt and asks the LLM to summarize it.
  ```python
  chain = load_summarize_chain(llm, chain_type="stuff")
  chain.run(docs)
  ```

- **`map_reduce`**: Ideal for large documents. It splits the text into chunks, summarizes each independently (map step), then combines the summaries into a final result (reduce step).
  ```python
  chain = load_summarize_chain(llm, chain_type="map_reduce")
  chain.run(docs)
  ```

- **`refine`**: Best for iterative refinement. Starts with an initial summary and progressively improves it by incorporating new document chunks.
  ```python
  chain = load_summarize_chain(llm, chain_type="refine")
  chain.run(docs)
  ```

For PDFs or large files, always **split the text first** using `CharacterTextSplitter` or `RecursiveCharacterTextSplitter`. This ensures the input fits within token limits and improves performance.

> 💡 **Tip**: Use `map_reduce` for large documents and `stuff` for short ones. The `refine` method is excellent for dynamic or sequential content like novels or long reports.

For detailed examples and code, refer to the official LangChain documentation:  
👉 [LangChain Summarization Tutorial](https://js.langchain.com/docs/tutorials/summarization)  
👉 [LangChain Docs – Summarize Text](https://python.langchain.com/v0.2/docs/how_to/summarize_refine/)

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

load_dotenv()

True

In [2]:
## Initialize the ChatGroq model
model = ChatGroq(model="openai/gpt-oss-120b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7e5274dbb620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7e5274b94440>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

## getting the text from the speech.txt

with open("/home/prashant/Documents/gen-ai/speech.txt","rb") as f:
    speech = f.read().decode("utf-8")

In [7]:
## creating the prompt for summarization
prompt = [
    SystemMessage(content="You are a helpful assistant that summarizes the text."),
    HumanMessage(content=f"Please provide a short and concise summary of the following text: {speech}")
]

In [10]:
model.get_num_tokens(speech)

/home/prashant/.local/lib/python3.13/site-packages/langchain_core/language_models/base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


940

In [13]:
model.invoke(prompt)

AIMessage(content='The speaker argues that education is a fundamental human right, essential for individual potential, economic growth, and societal progress, and calls for universal, relevant, and inclusive access—through investment in schools, teachers, technology, and skill‑focused curricula. They urge immediate collective action to bridge the digital divide, break financial and cultural barriers, and empower every child to become a future innovator and leader. The speech concludes by linking this educational mission to broader struggles for justice and equality, citing historic orations—from Martin\u202fLuther\u202fKing\u202fJr. to Sojourner Truth—as exemplars of the transformative power of ideas.', additional_kwargs={'reasoning_content': 'The user asks: "Please provide a short and concise summary of the following text:" then includes a long speech about education, plus mentions of historic speeches. We need to summarize concisely. Provide a short summary. Should capture main point

### Prompt Template Text Summarization

In [16]:
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate

template="""
Write the summary of the follpwing speech in less than 100 words.
Speech: {speech}
Translate the precise summary to {language}.
"""

prompt=PromptTemplate(
    input_variables=["speech","language"],
    template=template
)

prompt

PromptTemplate(input_variables=['language', 'speech'], input_types={}, partial_variables={}, template='\nWrite the summary of the follpwing speech in less than 100 words.\nSpeech: {speech}\nTranslate the precise summary to {language}.\n')

In [18]:
final_prompt=prompt.format(speech=speech,language="Hindi")
final_prompt

'\nWrite the summary of the follpwing speech in less than 100 words.\nSpeech: My fellow citizens,\nToday, I want to talk about something that has the power to change lives, communities, and nations. I\'m talking about education.\nIn a world where knowledge is the new currency, education is the key to unlocking our full potential. \nIt\'s the foundation upon which we build our dreams, our careers, and our future.\nFor too long, we\'ve been told that education is a privilege, a luxury that only a few can afford. \nBut I say to you, education is a right, not a privilege. It\'s a fundamental human right that should be available to every child, regardless of their background, their zip code, or their bank balance.\nThink about it. Every child who is denied education is a future leader, a future innovator, a future game-changer who will never get the chance to realize their full potential.\nBut it\'s not just about individual potential. Education is the backbone of our economy, our democracy

In [19]:
model.get_num_tokens(final_prompt)

970

In [21]:
llm_chain=LLMChain(prompt=prompt, llm=model)
summary=llm_chain.invoke({"speech":speech,"language":"Hindi"})
summary

{'speech': 'My fellow citizens,\nToday, I want to talk about something that has the power to change lives, communities, and nations. I\'m talking about education.\nIn a world where knowledge is the new currency, education is the key to unlocking our full potential. \nIt\'s the foundation upon which we build our dreams, our careers, and our future.\nFor too long, we\'ve been told that education is a privilege, a luxury that only a few can afford. \nBut I say to you, education is a right, not a privilege. It\'s a fundamental human right that should be available to every child, regardless of their background, their zip code, or their bank balance.\nThink about it. Every child who is denied education is a future leader, a future innovator, a future game-changer who will never get the chance to realize their full potential.\nBut it\'s not just about individual potential. Education is the backbone of our economy, our democracy, and our society. It\'s the driving force behind innovation, prog

### Stuff Documents Chain - Text Summarization

In [23]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("./apjspeech.pdf")
docs=loader.load_and_split()
docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': './apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wea

In [34]:
## creating the template and prompt for summarization
template="""Write a concise summary of the following:{speech}"""

prompt=PromptTemplate(
    input_variables=["speech"],
    template=template
)

In [35]:
from langchain_classic.chains.summarize import load_summarize_chain

chain=load_summarize_chain(llm=model,
                           chain_type="stuff", 
                           prompt=prompt,
                           document_variable_name="speech", 
                           verbose=True)
summary=chain.invoke(docs)
summary



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have many uni

{'input_documents': [Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': './apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity wh

In [36]:
print(summary["output_text"])

**Summary of Dr. A.P.J. Abdul Kalam’s Departing Address (May 2006‑2007)**  

Dr. Kalam reflected on his five‑year tenure as President, thanking the nation’s diverse citizens and outlining the vision that would drive India toward a “Developed India 2020.” His key messages were:

| Theme | Core Message & Illustrations |
|-------|------------------------------|
| **Accelerate Development – Youth Aspiration** | A school‑girl’s question on why India cannot be developed by 2020 epitomises the millions of young Indians who demand a prosperous, safe, proud nation. Their dreams must guide all policies. |
| **Empower Villages** | Visits to empowered tribal councils (e.g., Khuza ma, Nagaland) showed that when villages control finances and have good road connectivity, they can become self‑sufficient and prosperous. |
| **Mobilise Rural Core Competence (PURA)** | The Periyar PURA model (65 villages, 300 000 people) integrates physical, electronic and knowledge connectivity, creating health, educati

### Map Reduce to Summarization Large Documents

In [37]:
docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': './apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wea

In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

final_doc = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100).split_documents(docs)
final_doc

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': './apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wea

In [48]:
## creating the template and prompt for summarization
n_template="""Write a concise summary of the following:
Speech: {text}
Summary:"""

mapreduce_prompt=PromptTemplate(
    input_variables=["text"],
    template=n_template
)

final_prompt="""
Provide the final summary of the entire speech with these important points.
Add a motivation title,start the precise summary with an introduction and provide the summary in number of
points for the speech.
Speech: {text}
"""

final_mapreduce_prompt=PromptTemplate(
    input_variables=["text"],
    template=final_prompt
)

In [50]:
from langchain_classic.chains.summarize import load_summarize_chain

mapreduce_chain=load_summarize_chain(llm=model,
                           chain_type="map_reduce", 
                           map_prompt=mapreduce_prompt,
                           combine_prompt=final_mapreduce_prompt,
                           verbose=True)
mapreduce_summary=mapreduce_chain.invoke(final_doc)
mapreduce_summary



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:
Speech: A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I 

{'input_documents': [Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': './apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity wh

In [51]:
print(mapreduce_summary["output_text"])

**Motivation Title – “Dream Big, Act Bold: Turning India’s Vision of a Developed Nation into Reality”**

---

### Introduction  
In his departing address, Dr A.P.J. Abdul Kalam thanks the nation for an unforgettable five‑year presidency, celebrates the energy of India’s 540 million‑strong youth, and lays out a clear, action‑oriented roadmap for a “Developed India 2020.” Drawing on personal encounters—from a curious schoolgirl in Haryana to resilient farmers, a disabled scholar, and disaster‑hit communities—he illustrates how courage, connectivity, and collaboration can convert aspirations into tangible progress across rural development, agriculture, technology, defence, and education.

---

## Precise Summary – Key Points  

1. **Gratitude & Youth as the Nation’s Wealth**  
   - Kalam thanks Indians at home and abroad and stresses that the future’s strength lies in the enthusiasm, ideas, and moral fibre of the youth.

2. **Ten Priority Messages for National Progress**  
   1. Accelerat

### Refine Chain Text Summarization

In [52]:
refine_chain=load_summarize_chain(llm=model,
                            chain_type="refine",
                            verbose=True)

refine_summary=refine_chain.invoke(final_doc)
refine_summary



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have man

{'input_documents': [Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': './apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity wh

In [53]:
print(refine_summary["output_text"])

**Refined Summary – A.P.J. Abdul Kalam’s Farewell Address (June 2007)  
(augmented with the “Distinctive‑Profile,” the nine‑point national vision, and the newly‑added concluding passage)**  

---

### 1.  The Ten‑Point Roadmap (with enriched examples & linkage to the nine‑point vision)

| # | Priority (Kalam’s wording) | Core Insight & Illustrative Example | How it Serves the **Nine‑Point National Vision** |
|---|-----------------------------|--------------------------------------|---------------------------------------------------|
| **1** | **Youth‑driven development** | >15 lakh young Indians expressed a single dream – a safe, prosperous, proud India by 2020. | **(9) Prosperous, peaceful, happy nation** – youth are the engine of growth and social harmony. |
| **2** | **Village empowerment** | • **Khumza, Nagaland** – tribal council with financial powers, abundant produce, but no quality roads.<br>• **6000+ farmers at Rashtrapati Bhavan** – toured thematic gardens, exchanged ideas wi